In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
yield_model = joblib.load("../models/yield_model.pkl")
price_model = joblib.load("../models/price_model.pkl")
soil_model = joblib.load("../models/soil_model.pkl")

print("Models loaded successfully")

Models loaded successfully


In [3]:
df = pd.read_csv("../data/final_dataset.csv")

df.head()

,State,District_Name,Year,Season,Crop,Area,Production,Rainfall,Temperature,Yield
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2763.2,25.53,1.594896
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2596.8,25.53,1.594896
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2487.0,25.53,1.594896
3,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2363.9,25.53,1.594896
4,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2908.0,25.53,1.594896


In [5]:
le_state = joblib.load("../models/state_encoder.pkl")
le_crop = joblib.load("../models/crop_encoder.pkl")
le_season = joblib.load("../models/season_encoder.pkl")

df["State"] = le_state.transform(df["State"])
df["Crop"] = le_crop.transform(df["Crop"])
df["Season"] = le_season.transform(df["Season"])

In [ ]:
'''
# ---------------- old code-----------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import joblib

df = pd.read_csv("data/final_dataset.csv")

# Remove missing values
df = df.dropna()

# Safety check
print("Missing values:")
print(df.isnull().sum())

df = df.drop_duplicates()

# Sample dataset to reduce size (optional but recommended)
# df = df.sample(100000, random_state=42)

print("Dataset size after sampling:", df.shape)

# Apply log transformation to target variable
df["Yield"] = np.log1p(df["Yield"])


print(df.head())

le_state = LabelEncoder()
le_crop = LabelEncoder()
le_season = LabelEncoder()

df["State"] = le_state.fit_transform(df["State"])
df["Crop"] = le_crop.fit_transform(df["Crop"])
df["Season"] = le_season.fit_transform(df["Season"])

df = df.drop(columns=["District_Name"])

features = [
    "State",
    "Crop",
    "Season",
    "Area",
    "Rainfall",
    "Temperature"
]

X = df[features]
y = df["Yield"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {

    # "Linear Regression": LinearRegression(),

    # "Random Forest": RandomForestRegressor(
    #      n_estimators=500,
    # max_depth=25,
    # min_samples_split=5,
    # min_samples_leaf=2,
    # n_jobs=-1,
    # random_state=42
    # ),

    # "Gradient Boosting": GradientBoostingRegressor(),

    "XGBoost": XGBRegressor(
        n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
    )
}

results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results[name] = (rmse, r2)

    print(f"{name}")
    print("RMSE:", rmse)
    print("R2:", r2)
    print("---------------------")
    
for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="r2"
    )

    print(name, "CV R2:", scores.mean()) 
    
best_model_name = max(results, key=lambda x: results[x][1])
best_model = models[best_model_name]

print("Best Model:", best_model_name)

joblib.dump(best_model, "models/yield_model.pkl")  

print(df["Crop"].nunique())
    
 '''

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error , mean_squared_error, r2_score

from xgboost import XGBRegressor

import joblib
from sklearn.model_selection import GroupKFold


# -------------------------------
# Load Dataset
# -------------------------------

df = pd.read_csv("../data/final_dataset.csv")

print("Original dataset size:", df.shape)


# -------------------------------
# Data Cleaning
# -------------------------------

df = df.dropna()
df = df.drop_duplicates()

print("Dataset after cleaning:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())


# -------------------------------
# Log transform target variable
# -------------------------------

df["Yield"] = np.log1p(df["Yield"])


# -------------------------------
# Encode categorical variables
# -------------------------------

le_state = LabelEncoder()
le_crop = LabelEncoder()
le_season = LabelEncoder()

df["State"] = le_state.fit_transform(df["State"])
df["Crop"] = le_crop.fit_transform(df["Crop"])
df["Season"] = le_season.fit_transform(df["Season"])


# Drop high-cardinality column
df = df.drop(columns=["District_Name"])


# -------------------------------
# Feature Selection
# -------------------------------

features = [
    "State",
    "Crop",
    "Season",
    "Area",
    "Rainfall",
    "Temperature"
]

X = df[features]
y = df["Yield"]


# -------------------------------
# Train Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# -------------------------------
# Model Definition
# -------------------------------

models = {

    "XGBoost": XGBRegressor(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=10,
        subsample=0.9,
        colsample_bytree=0.9,
        gamma=0.1,
        min_child_weight=3,
        random_state=42,
        n_jobs=-1
    )
}


# -------------------------------
# Model Training
# -------------------------------

results = {}
trained_models = {}

for name, model in models.items():

    print("\nTraining:", name)

    model.fit(X_train, y_train)

    trained_models[name] = model

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results[name] = (rmse, r2)

    print("MAE:", mae)
    print("MSE:", mse)
    print("RMSE:", rmse)
    print("R2 Score:", r2)


# -------------------------------
# Cross Validation
# -------------------------------

print("\nGroup Cross Validation Results")

gkf = GroupKFold(n_splits=5)

for name, model in models.items():

    scores = cross_val_score(
    model,
    X,
    y,
    cv=gkf.split(X, y, groups=df["State"]),
    scoring="r2"
    )

    print(name, "Group CV R2:", scores.mean())


# -------------------------------
# Select Best Model
# -------------------------------

best_model_name = max(results, key=lambda x: results[x][1])
best_model = trained_models[best_model_name]

print("\nBest Model:", best_model_name)


# -------------------------------
# Save Model
# -------------------------------

joblib.dump(best_model, "../models/yield_model.pkl")

print("Model saved successfully at models/yield_model.pkl")

# -------------------------------
# Save Encoders
# -------------------------------

joblib.dump(le_state, "models/state_encoder.pkl")
joblib.dump(le_crop, "models/crop_encoder.pkl")
joblib.dump(le_season, "models/season_encoder.pkl")

print("Encoders saved successfully")


# -------------------------------
# Dataset Information
# -------------------------------

print("\nTotal unique crops:", df["Crop"].nunique())
    

Original dataset size: (8724996, 10)
Dataset after cleaning: (8673230, 10)

Missing values:
State            0
District_Name    0
Year             0
Season           0
Crop             0
Area             0
Production       0
Rainfall         0
Temperature      0
Yield            0
dtype: int64

Training: XGBoost
MAE: 0.12137982840874871
MSE: 0.0362683464244631
RMSE: 0.19044250162309645
R2 Score: 0.9662152404241321

Group Cross Validation Results
XGBoost Group CV R2: 0.43712370095064285

Best Model: XGBoost


FileNotFoundError: [Errno 2] No such file or directory: 'models/yield_model.pkl'

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor

import joblib


# ------------------------------
# Load dataset
# ------------------------------

# df = pd.read_csv("data/price_data.csv")
df = pd.read_csv("../data/price_data.csv")

# remove hidden spaces in column names
df.columns = df.columns.str.strip()

print(df.columns)

print("Original dataset shape:", df.shape)


# ------------------------------
# Data Cleaning
# ------------------------------

df = df.dropna()
df = df.drop_duplicates()

print("Dataset after cleaning:", df.shape)


# ------------------------------
# Rename columns for convenience
# ------------------------------

df = df.rename(columns={
    "Wholesale_Price[Rs. Per Quintal]": "Price"
    
    
})

# df = df.rename(columns={
#     "CROP_PRICE": "Price",
#      "STATE" : "State",
#      "CROP" : "Crop"
     
    
    
# })


# ------------------------------
# Log transform target variable
# ------------------------------

df["Price"] = np.log1p(df["Price"])


# ------------------------------
# Encode categorical variables
# ------------------------------

le_state = LabelEncoder()
le_crop = LabelEncoder()

df["State"] = le_state.fit_transform(df["State"])
df["Crop"] = le_crop.fit_transform(df["Crop"])



# ------------------------------
# Feature Selection
# ------------------------------

features = [
    "Year",
    "Month",
    "State",
    "Crop",
    "Temperature (Celsis)",
    "Rainfall in mm"
    # "TEMPERATURE",
    # "RAINFALL"
]

X = df[features]
y = df["Price"]


# ------------------------------
# Train Test Split
# ------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ------------------------------
# Model Definition
# ------------------------------

model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)


# ------------------------------
# Model Training
# ------------------------------

print("\nTraining Price Prediction Model...")

model.fit(X_train, y_train)

pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\nModel Performance")
print("RMSE:", rmse)
print("R2 Score:", r2)


# ------------------------------
# Cross Validation
# ------------------------------

scores = cross_val_score(
    model,
    X,
    y,
    cv=3,
    scoring="r2"
)

print("\nCross Validation R2:", scores.mean())


# ------------------------------
# Save model
# ------------------------------

joblib.dump(model, "../models/price_model.pkl")

print("\nPrice model saved at models/price_model.pkl")

Index(['Year', 'Month', 'State', 'Crop', 'Wholesale_Price[Rs. Per Quintal]',
       'Temperature (Celsis)', 'Rainfall in mm'],
      dtype='object')
Original dataset shape: (2280, 7)
Dataset after cleaning: (2280, 7)

Training Price Prediction Model...

Model Performance
RMSE: 0.1603173980482706
R2 Score: 0.9544180694188061

Cross Validation R2: 0.9102346730089058

Price model saved at models/price_model.pkl


In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib

df = pd.read_csv("../data/soil_crop_dataset.csv")

# -------------------------------
# Encoding
# -------------------------------

le_state = LabelEncoder()
le_soil = LabelEncoder()
le_crop = LabelEncoder()

df["STATE"] = le_state.fit_transform(df["STATE"])
df["SOIL_TYPE"] = le_soil.fit_transform(df["SOIL_TYPE"])
df["CROP"] = le_crop.fit_transform(df["CROP"])

# -------------------------------
# Features and Target
# -------------------------------

X = df[[
    "STATE",
    "SOIL_TYPE",
    "N_SOIL",
    "P_SOIL",
    "K_SOIL",
    "TEMPERATURE",
    "HUMIDITY",
    "ph",
    "RAINFALL"
]]

y = df["CROP"]

# -------------------------------
# Train Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# Model Training
# -------------------------------

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    random_state=42
)

model.fit(X_train, y_train)

# -------------------------------
# Prediction
# -------------------------------

y_pred = model.predict(X_test)

# -------------------------------
# Evaluation Metrics
# -------------------------------

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred, average="weighted")

recall = recall_score(y_test, y_pred, average="weighted")

f1 = f1_score(y_test, y_pred, average="weighted")

cm = confusion_matrix(y_test, y_pred)

print("\nSoil Model Evaluation")
print("---------------------------")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nConfusion Matrix:\n", cm)

# -------------------------------
# Save Model
# -------------------------------

joblib.dump(model, "../models/soil_model.pkl")

joblib.dump(le_state, "../models/soil_state_encoder.pkl")
joblib.dump(le_soil, "../models/soil_encoder.pkl")
joblib.dump(le_crop, "../models/soil_crop_encoder.pkl")

print("\nSoil recommendation model saved")

d:\python2024\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\python2024\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Soil Model Evaluation
---------------------------
Accuracy: 0.05454545454545454
Precision: 0.03277574312678677
Recall: 0.05454545454545454
F1 Score: 0.03736154650957893

Confusion Matrix:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Soil recommendation model saved
